# 05 · LLM layer on top of the engine
The LLM (a) translates plain English into our query language, (b) writes an answer from the top-5 retrieved abstracts, and (c) every answer sentence is checked by our tf-idf citation checker. Without an API key the parser falls back to rules and no answer is generated.

In [ ]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
from scholar_ir.config import load_config, p
cfg = load_config()  # config.yaml at the repo root (or $SCHOLAR_IR_CONFIG)
R = p(cfg, "results_dir")

In [ ]:
from scholar_ir.engine import SearchEngine
engine = SearchEngine.load(cfg)
index, papers = engine.index, engine.papers
print(f"{index.N:,} papers, years {papers.year.min()}-{papers.year.max()}, tuned params loaded: {bool(engine.tuned)}")

In [ ]:
from scholar_ir.llm_client import LLMClient
from scholar_ir.llm_query_parser import llm_parse, rule_parse
client = LLMClient(cfg); print(client.provider, client.model, 'available:', client.available())
req = 'recent papers on retrieval-augmented generation for question answering since 2022'
sq, src = llm_parse(req, client, int(engine.years.max())); print(src, sq.model_dump()); print('compiled:', sq.compile())
print('rule parser:', rule_parse(req, int(engine.years.max())).compile())

In [ ]:
from scholar_ir.citation_checker import CitationChecker
from scholar_ir.rag import answer
ck = engine.cfg['checker']; checker = CitationChecker(index, ck['alpha'], ck['tau'])
a = answer('How does dense passage retrieval differ from sparse retrieval such as BM25?', engine, client, checker)
print(a.report())

## The checker on a deliberately weak sentence and a fabricated citation

In [ ]:
ctx = a.context; first = list(ctx)[0]
for s in [f'Dense retrieval models were first proposed for protein folding in 1995 [arXiv:{first}].',
          'Sparse retrieval is always better than dense retrieval [arXiv:2101.00001].']:
    c = checker.check_sentence(s, ctx); print(f'{s}\n   support={c.support} flags={c.flags}\n')

## Measured results (from `llm_experiments`)

In [ ]:
pd.read_csv(R / 'llm/parser_eval.csv') if (R / 'llm/parser_eval.csv').exists() else 'run: python -m scholar_ir.llm_experiments parser'

In [ ]:
c = R / 'llm/checker_eval.csv'
pd.read_csv(c) if c.exists() else 'run: python -m scholar_ir.llm_experiments checker (after labelling)'

In [ ]:
e = R / 'llm/checker_errors.csv'
pd.read_csv(e).head(10) if e.exists() else None